In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, ConfusionMatrixDisplay)

os.makedirs("/analytics", exist_ok=True)
# Primary load from Seaborn; fallback to local CSV if internet is offline
csv_path = "D:\analytics\titanic.csv"
try:
    df = sns.load_dataset("titanic")
    df.to_csv(csv_path, index=False)
    print("Dataset loaded from Seaborn and saved to /analytics/titanic.csv")
except Exception as e:
    print(f"Internet load failed ({e}). Loading from local fallback...")
    df = pd.read_csv(csv_path)

# Profile the dataset
print("\n--- DATA PROFILE ---")
print(f"Shape: {df.shape}")
print("\n--- Info ---")
df.info()
print("\n--- Summary Statistics ---")
print(df.describe(include="all"))

# Compute missing value percentages
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing_counts, 'Missing_Pct': missing_pct})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Missing_Pct', ascending=False)

print("\n--- Missing Value Percentages ---")
print(missing_df)

df_clean = df.copy()

# 1. Drop high missing column (>30%)
df_clean = df_clean.drop(columns=['deck'])

# 2. Drop rows for variables missing < 5%
df_clean = df_clean.dropna(subset=['embarked', 'embark_town'])

# 3. Impute median for variables missing 5% - 30%
age_median = df_clean['age'].median()
df_clean['age'] = df_clean['age'].fillna(age_median)

print(f"\nRemaining missing values after cleaning: {df_clean.isnull().sum().sum()}")
print(f"Cleaned shape: {df_clean.shape}")

def count_outliers_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return len(outliers), lower_bound, upper_bound

age_outliers, age_lb, age_ub = count_outliers_iqr(df_clean['age'])
fare_outliers, fare_lb, fare_ub = count_outliers_iqr(df_clean['fare'])

print("\n--- IQR OUTLIER REPORT ---")
print(f"Age Outliers: {age_outliers} (Bounds: [{age_lb:.2f}, {age_ub:.2f}])")
print(f"Fare Outliers: {fare_outliers} (Bounds: [{fare_lb:.2f}, {fare_ub:.2f}])")

# Summary Stats for Fare
fare_mean = df_clean['fare'].mean()
fare_median = df_clean['fare'].median()
fare_mode = df_clean['fare'].mode()[0]

print("\n--- FARE DISTRIBUTION STATS ---")
print(f"Mean:   {fare_mean:.4f}")
print(f"Median: {fare_median:.4f}")
print(f"Mode:   {fare_mode:.4f}")

# Written skewness justification:
# Fare ordering: Mean (32.09) > Median (14.45) > Mode (8.05).
# Because Mean > Median > Mode, the Fare distribution is strongly right-skewed.

# Plots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.histplot(df_clean['age'], kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Age Distribution (Histogram)')

sns.boxplot(x=df_clean['age'], ax=axes[0, 1], color='skyblue')
axes[0, 1].set_title('Age Boxplot')

sns.histplot(df_clean['fare'], kde=True, ax=axes[1, 0], color='salmon')
axes[1, 0].set_title('Fare Distribution (Histogram)')

sns.boxplot(x=df_clean['fare'], ax=axes[1, 1], color='salmon')
axes[1, 1].set_title('Fare Boxplot')

plt.tight_layout()
plt.show()

# Boolean Masking Survival Calculations
survived_total = df_clean['survived']

# (a) Sex
rate_female = df_clean[df_clean['sex'] == 'female']['survived'].mean()
rate_male = df_clean[df_clean['sex'] == 'male']['survived'].mean()

# (b) Pclass
rate_p1 = df_clean[df_clean['pclass'] == 1]['survived'].mean()
rate_p2 = df_clean[df_clean['pclass'] == 2]['survived'].mean()
rate_p3 = df_clean[df_clean['pclass'] == 3]['survived'].mean()

print("\n--- SURVIVAL RATES (BOOLEAN MASKING) ---")
print(f"By Sex    -> Female: {rate_female:.4f} | Male: {rate_male:.4f}")
print(f"By Pclass -> Class 1: {rate_p1:.4f} | Class 2: {rate_p2:.4f} | Class 3: {rate_p3:.4f}")

# (c) Sex AND Pclass combined using bitwise &
print("\nBy Sex AND Pclass:")
for s in ['female', 'male']:
    for c in [1, 2, 3]:
        mask = (df_clean['sex'] == s) & (df_clean['pclass'] == c)
        rate = df_clean[mask]['survived'].mean()
        print(f"  {s.capitalize()}, Class {c}: {rate:.4f}")

# 6x6 Correlation Matrix (Excluding adult_male and alone)
corr_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
corr_matrix = df_clean[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("6x6 Correlation Matrix")
plt.show()

# Rank top 2 strongest off-diagonal pairs
corr_pairs = (
    corr_matrix.abs().unstack()
    .reset_index()
    .rename(columns={0: 'abs_corr'})
)
# Filter out self-correlations (where feature_1 == feature_2)
corr_pairs = corr_pairs[corr_pairs['level_0'] != corr_pairs['level_1']]
# Drop duplicates (A-B vs B-A)
corr_pairs['pair'] = corr_pairs.apply(lambda r: tuple(sorted([r['level_0'], r['level_1']])), axis=1)
unique_pairs = corr_pairs.drop_duplicates(subset=['pair']).sort_values(by='abs_corr', ascending=False)

print("\nTop Off-Diagonal Correlations:")
for idx, row in unique_pairs.head(2).iterrows():
    p1, p2 = row['pair']
    val = corr_matrix.loc[p1, p2]
    print(f"  {p1} vs {p2}: Correlation = {val:.4f} (Abs = {row['abs_corr']:.4f})")

# Chart 1: Barplot - Survival Rate by Sex and Pclass
plt.figure(figsize=(7, 5))
sns.barplot(data=df_clean, x='pclass', y='survived', hue='sex', ci=None, palette='Set1')
plt.title("1. Survival Rate by Passenger Class and Sex")
plt.ylabel("Survival Rate")
plt.show()
"""
This chart highlights the stark intersection of gender and class privilege in survival outcomes.
Women in 1st and 2nd class had survival rates above 85%, whereas men in 3rd class had a survival rate below 15%.
"""

# Chart 2: Boxplot - Fare Distribution by Survival and Class
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_clean, x='pclass', y='fare', hue='survived', palette='Set2', showfliers=False)
plt.title("2. Fare Distribution by Class and Survival Status (Outliers Hidden)")
plt.ylabel("Fare")
plt.show()
"""
Across 1st class passengers, those who survived generally paid higher median fares than those who died.
In 2nd and 3rd classes, fare variations were minor, but 1st class survivors show a distinct upward skew in purchasing power.
"""

# Chart 3: Heatmap - Survival Rates by Age Bins and Sex
df_clean['age_bin'] = pd.cut(df_clean['age'], bins=[0, 12, 18, 35, 60, 100], labels=['Child', 'Teens', 'Young Adult', 'Adult', 'Senior'])
age_sex_pivot = df_clean.pivot_table(index='age_bin', columns='sex', values='survived', aggfunc='mean')

plt.figure(figsize=(6, 4))
sns.heatmap(age_sex_pivot, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("3. Survival Rate by Age Group and Sex")
plt.show()
"""
Children (ages 0-12) experienced significantly higher survival rates compared to older age groups, regardless of sex.
Male survival drops dramatically past childhood, bottoming out for adult and senior men.
"""

# Chart 4: Categorical Pointplot - Survival by Family Size and Class
df_clean['family_size'] = df_clean['sibsp'] + df_clean['parch'] + 1
plt.figure(figsize=(8, 5))
sns.pointplot(data=df_clean, x='family_size', y='survived', hue='pclass', ci=None, markers='o')
plt.title("4. Survival Rate by Family Size and Passenger Class")
plt.xlabel("Family Size (SibSp + Parch + 1)")
plt.ylabel("Survival Rate")
plt.show()
"""
Solo travelers (family size = 1) and very large families (family size > 4) suffered lower survival rates across all classes.
Small families of 2 to 4 members consistently achieved higher survival rates, particularly in 1st and 2nd class.
"""
# Manual Z-Score calculation: z = (x - mean) / std
df_clean['age_z_manual'] = (df_clean['age'] - df_clean['age'].mean()) / df_clean['age'].std()
df_clean['fare_z_manual'] = (df_clean['fare'] - df_clean['fare'].mean()) / df_clean['fare'].std()

# Verification using StandardScaler
scaler = StandardScaler()
scaled_vals = scaler.fit_transform(df_clean[['age', 'fare']])
df_clean['age_z_scaler'] = scaled_vals[:, 0]
df_clean['fare_z_scaler'] = scaled_vals[:, 1]

print("\n--- STANDARDIZATION SANITY CHECK ---")
print("Original Age  -> Mean:", round(df_clean['age'].mean(), 4), "| Std:", round(df_clean['age'].std(), 4))
print("Z-Scaled Age  -> Mean:", round(df_clean['age_z_manual'].mean(), 4), "| Std:", round(df_clean['age_z_manual'].std(), 4))

print("Original Fare -> Mean:", round(df_clean['fare'].mean(), 4), "| Std:", round(df_clean['fare'].std(), 4))
print("Z-Scaled Fare -> Mean:", round(df_clean['fare_z_manual'].mean(), 4), "| Std:", round(df_clean['fare_z_manual'].std(), 4))

# Overlaid Distribution Comparison Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.kdeplot(df_clean['age'], ax=axes[0], label='Original Age', color='blue')
sns.kdeplot(df_clean['age_z_manual'], ax=axes[0], label='Standardized Age (z)', color='green')
axes[0].set_title("Age Transformation Overlay")
axes[0].legend()

sns.kdeplot(df_clean['fare'], ax=axes[1], label='Original Fare', color='red')
sns.kdeplot(df_clean['fare_z_manual'], ax=axes[1], label='Standardized Fare (z)', color='orange')
axes[1].set_title("Fare Transformation Overlay")
axes[1].legend()

plt.tight_layout()
plt.show()